# CPSC 4830 – Final Exam (Amazon Review Assistant with GPT)

**Duration:** 3 Hours  
**Total Marks:** 100  

This notebook is your **exam paper**. All answers must be written in this notebook.

---

## IMPORTANT INSTRUCTIONS

- You **must use Google Colab** on the **exam room PCs**.  
  - **Personal laptops are not allowed.** No exceptions.
- You **may NOT** use ChatGPT, Copilot, Claude, or any other external AI assistant.
- You **may** use:
  - The course materials and examples on **D2L**  
  - The official **Hugging Face `datasets` documentation**: <https://huggingface.co/docs/datasets>  
  - The official **OpenAI API documentation**: <https://platform.openai.com/docs/api-reference>  
- Do **not** hard-code your real OpenAI API key into the notebook you submit.  
  - Use environment variables / Colab secrets, and remove the key before uploading.

**Submission:**  
- When you are done, make sure all cells have been run.  
- Download the `.ipynb` from Colab and upload it to D2L under the Final Exam submission.

---


## Marking Overview (Rubric Summary)

- **Part 1 – Load, Explore, and Sample Reviews:** 25 marks  
- **Part 2 – GPT Summarization of Reviews:** 35 marks  
- **Part 3 – GPT Sentiment Classification of Summaries:** 25 marks  
- **Part 4 – Mini “Review Assistant” Chatbot:** 15 marks  

Total = **100 marks**.

Detailed mark breakdown is included in each part below.


---

## Part 1 – Load, Explore, and Sample Reviews (25 marks)

We will use the **public `amazon_polarity` dataset** from Hugging Face.

> Dataset: `amazon_polarity`  
> Columns of interest:  
> • `label` – 0 = negative, 1 = positive  
> • `content` – review text

Work on a manageable subset (for example, the first 20,000 rows) so that Colab runs smoothly.

### Part 1.1 – Load a review split (10 marks)

**Goal:** Load the dataset and inspect the first rows.

**What to do:**
- Use `datasets.load_dataset("amazon_polarity", split="train[:20000]")` (or similar).  
- Display the first **10 rows**, showing at least:
  - the **label** column  
  - the **content** column (review text)

**Marks (10):**
- Correct dataset loading (5)  
- Correct display of 10 rows with the required columns (5)

---

### Part 1.2 – Label distribution (10 marks)

**Goal:** Understand how labels are distributed.

**What to do:**
- Compute and print a **frequency table** and **percentage breakdown** of `label` values (0 and 1).  
- Add a **short markdown explanation (2–3 sentences)** commenting on whether the data is balanced or skewed.

**Marks (10):**
- Correct frequency & percentage calculation (6)  
- Clear commentary about balance / skew (4)

---

### Part 1.3 – Balanced sample (5 marks)

**Goal:** Build a balanced subset called `balanced_reviews`.

**What to do:**
- From the loaded split, create a subset of **1,000 reviews** with **exactly 500 label=0 and 500 label=1** (sample with replacement if needed).  
- Show the new label distribution for `balanced_reviews`.

**Marks (5):**
- Balanced sampling logic (3)  
- Correct distribution printout (2)


### Part 1.1 – Starter code for dataset loading (you may modify as needed)

In [ ]:
# Install required packages (run once at the top of your notebook in Colab)
!pip install -q datasets pandas

from datasets import load_dataset
import pandas as pd

# TODO: Load a manageable subset of the amazon_polarity dataset.
# Hint: use split="train[:20000]"

dataset = load_dataset(
    "amazon_polarity",
    split="train[:20000]"
)

print(dataset)

df = dataset.to_pandas()
print(df.head(10)[["label", "content"]])  # you may adjust this


Dataset({
    features: ['label', 'title', 'content'],
    num_rows: 20000
})
   label                                            content
0      1  This sound track was beautiful! It paints the ...
1      1  I'm reading a lot of reviews saying that this ...
2      1  This soundtrack is my favorite music of all ti...
3      1  I truly like this soundtrack and I enjoy video...
4      1  If you've played the game, you know how divine...
5      1  I am quite sure any of you actually taking the...
6      0  This is a self-published book, and if you want...
7      1  I loved Whisper of the wicked saints. The stor...
8      1  I just finished reading Whisper of the Wicked ...
9      1  This was a easy to read book that made me want...


In [ ]:
# Part 1.2 – Compute label distribution
# TODO: Compute counts and percentages for each label value (0 and 1).

display(df.describe())
# display(df.columns)
# display(df.head(5))
df_count = df["label"].value_counts()
display(df_count)
df_percentage = df["label"].value_counts(normalize=True)*100
display(df_percentage)

,label
count,20000.000000
mean,0.512850
std,0.499847
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


,count
label,
1,10257
0,9743


,proportion
label,
1,51.285
0,48.715


In [ ]:
# Part 1.3 – Create balanced_reviews subset
# TODO: Sample exactly 500 rows with label=0 and 500 rows with label=1 (with replacement if needed).
# Store result in a variable named balanced_reviews.

balanced_reviews_0 = df[df["label"] == 0].sample(n=500, replace=True)
balanced_reviews_1 = df[df["label"] == 1].sample(n=500, replace=True)
display(balanced_reviews_0.shape)
display(balanced_reviews_1.shape)
# balanced_reviews = balanced_reviews_1.merge(balanced_reviews_0, how="outer")
# display(balanced_reviews.shape)
# balanced_reviews = balanced_reviews.append(df[df["label"] == 1].sample(n=500, replace=True))
# Combine the two samples vertically
balanced_reviews = pd.concat([balanced_reviews_0, balanced_reviews_1], axis=0)

# Shuffle the combined dataframe to mix labels
balanced_reviews = balanced_reviews.sample(frac=1, random_state=42).reset_index(drop=True)

# Display the shape and label distribution to verify
display(balanced_reviews.shape)
display(balanced_reviews['label'].value_counts())
display(balanced_reviews.head(5))


(500, 3)

(500, 3)

(1000, 3)

,count
label,
1,500
0,500


,label,title,content
0,1,I love my TiVo,This is the second TiVo DVR I've owned. I real...
1,1,Great Cd,I just got this cd a few days ago and I must s...
2,1,"A YOUNG, SURPRISING HERO","Hey, all of you people who are interested in r..."
3,1,Appliance 20 amp timer,Great timer for dedicated appliance circuit. I...
4,0,Dumb movie,This was the biggest waste of time I have ever...


_Use the next markdown cell to briefly describe your observations about the label distribution._

**Part 1.2 / 1.3 – Short commentary (2–3 sentences):**  
 I can see in the count that the dataset  when i count by  label (1,2), the presence of label 1 is  a little bigger 10257,vs the label 0 9743,so for a correct analysis we mus try to get balanced over the sampling, also looks like thes dataset similar to  spam where the label can classify the messages can  be calssified as spam or not, we cat be sure at this point.

---

## Part 2 – GPT Summarization of Reviews (35 marks)

In this part, you will use the **OpenAI chat API** to summarize review text.

Use a small, cost-effective model (for example `gpt-4.1-mini` or a model specified by your instructor).  
Refer to the OpenAI API docs: <https://platform.openai.com/docs/api-reference/chat>

We will summarize the `content` column of `balanced_reviews`.

### Part 2.1 – Summarization function (15 marks)

Write a function `summarize_review(text: str) -> str` that:

- Calls the **OpenAI chat completion** API using a **system** + **user** message pattern.  
- Asks the model to produce a **concise, neutral summary** in **30–50 words**, based only on the review text.  
- Returns the summary as a plain string.

**Marks (15):**
- Correct use of OpenAI chat API (8)  
- Clear summarization prompt with word limit (5)  
- Robustness (e.g., basic error handling / stripping whitespace) (2)

---

### Part 2.2 – Apply summarization to reviews (10 marks)

Apply `summarize_review` to (at least) the **first 100 rows** of `balanced_reviews`.

- Create a new column `summary` containing the GPT output.  
- Show a small table with **5 examples** that includes:
  - original `content`  
  - `label`  
  - `summary`

**Marks (10):**
- Successful application to ~100 reviews (6)  
- Clear table of 5 example rows (4)

---

### Part 2.3 – Save summarized data (10 marks)

Create a **Pandas DataFrame** with at least:

- `review_id` (or index)  
- `label`  
- `summary`  

Save it as **`summarized_reviews.csv`**.

**Marks (10):**
- Correct DataFrame construction (5)  
- File saved and path shown (5)


In [171]:
# Part 2.1 – Setup OpenAI client and define summarization function
# Hints:
# - Use environment variables or Colab's secret storage for your API key.
# - Use the chat completions API with a system + user message.

# n this part, you will use the OpenAI chat API to summarize review text.

# Use a small, cost-effective model (for example gpt-4.1-mini or a model specified by your instructor).
# Refer to the OpenAI API docs: https://platform.openai.com/docs/api-reference/chat

# We will summarize the content column of balanced_reviews.

# Part 2.1 – Summarization function (15 marks)
# Write a function summarize_review(text: str) -> str that:

# Calls the OpenAI chat completion API using a system + user message pattern.
# Asks the model to produce a concise, neutral summary in 30–50 words, based only on the review text.
# Returns the summary as a plain string.
# Marks (15):

# Correct use of OpenAI chat API (8)
# Clear summarization prompt with word limit (5)
# Robustness (e.g., basic error handling / stripping whitespace) (2)


# TODO: import and configure OpenAI client
# from openai import OpenAI
# client = OpenAI()

# TODO: define summarize_review(text: str) -> str
import os
import openai


from openai import OpenAI
from google.colab import userdata
apikey=userdata.get('OPENAI_API_KEY')
# print(apikey)
# openai.api_key = apikey

from openai import OpenAI
client = OpenAI(api_key=apikey)
model="gpt-4.1-mini",

def summarize_review(text: str):

  system_message = {
        "role": "system",
        "content": (
            "You are a helpful assistant. "
            "Please provide a concise, neutral summary of the given review text "
            "in 30 to 50 words, based only on the content provided."
        )
    }
  user_message = {
        "role": "user",
        "content": text
    }

  messages=[
    {"role": "developer", "content": "You are a helpful assistant."},
    # {"role" : "developer",  "content": (
    #         "You are a helpful assistant. "
    #         "Please provide a concise, neutral summary of the given review text "
    #         "in 30 to 50 words, based only on the content provided."
    #     )},
    {"role": "user", "content": text}
  ]
  try:
    completion = client.chat.completions.create(
      model="gpt-4.1-mini",
      temperature=0.1,
      max_tokens=250,
      messages=[system_message, user_message]

)


    return completion.choices[0].message.content
  except Exception as e:
    print(e)
    return ""


question="may i pass  if i solve  3 points of  6 the finals of cspc 4830"

question="describe an a merican bully"
test_function=summarize_review(question)
display(test_function)

'The American Bully is a muscular, compact dog breed known for its strength, loyalty, and friendly temperament. It has a broad head, strong build, and comes in various sizes. Despite its tough appearance, it is affectionate, good with families, and requires regular exercise and socialization.'

In [ ]:
# Part 2.2 – Apply summarization to at least 100 reviews
# TODO: apply summarize_review to the first 100 rows of balanced_reviews and store in a 'summary' column.
variable_magic_number_view=100
balanced_reviews['summary'] = balanced_reviews[:variable_magic_number_view]['content'].apply(summarize_review)

# Display the DataFrame with summaries
display(balanced_reviews.head(variable_magic_number_view))

# print(response)


,label,title,content,summary
0,1,I love my TiVo,This is the second TiVo DVR I've owned. I real...,The reviewer appreciates their second TiVo DVR...
1,1,Great Cd,I just got this cd a few days ago and I must s...,The reviewer recently purchased the CD and fin...
2,1,"A YOUNG, SURPRISING HERO","Hey, all of you people who are interested in r...",The reviewer read this Old English literature ...
3,1,Appliance 20 amp timer,Great timer for dedicated appliance circuit. I...,This timer is suitable for dedicated appliance...
4,0,Dumb movie,This was the biggest waste of time I have ever...,The reviewer found the movie to be a complete ...
...,...,...,...,...
95,0,He's not quite there yet...,Apart from some reasonable songs (One Wish - T...,The reviewer felt the album was lacking overal...
96,1,Enjoyable Read,This book is engaging and really brings the st...,The book is engaging and vividly brings the st...
97,0,Don't use this company never got item,This company sucks. It's been a lil over a mon...,"The customer is very dissatisfied, having not ..."
98,0,What happened?,"Clay knocked my socks off on American Idol, an...",The reviewer praises Clay's impressive perform...


In [ ]:
# Part 2.2 – Show 5 example summaries
# TODO: display 5 rows with [content, label, summary]

display(balanced_reviews.head(5)[["content", "label", "summary"]])



,content,label,summary
0,This is the second TiVo DVR I've owned. I real...,1,The reviewer appreciates their second TiVo DVR...
1,I just got this cd a few days ago and I must s...,1,The reviewer recently purchased the CD and fin...
2,"Hey, all of you people who are interested in r...",1,The reviewer read this Old English literature ...
3,Great timer for dedicated appliance circuit. I...,1,This timer is suitable for dedicated appliance...
4,This was the biggest waste of time I have ever...,0,The reviewer found the movie to be a complete ...


In [ ]:
# Part 2.3 – Save summarized data
# TODO: build a DataFrame with [review_id/index, label, summary] and save to 'summarized_reviews.csv'
balanced_reviews.to_csv('summarized_reviews.csv', index=False)


---

## Part 3 – GPT Sentiment Classification of Summaries (25 marks)

Now you will classify the sentiment of the **summaries**, using GPT again.

### Part 3.1 – Sentiment function (15 marks)

Write a function `classify_sentiment(summary: str) -> str` that:

- Uses the OpenAI chat API with a prompt such as:  
  *"Read this summarized review and classify its overall sentiment as one of: Positive, Neutral, or Negative. Return only the label."*  
- Returns exactly one of: `"Positive"`, `"Neutral"`, `"Negative"`.

Apply this function to all rows in `summarized_reviews` (or at least the same 100 rows), storing the result in a new column `sentiment_label`.

**Marks (15):**
- Correct API usage and prompt (8)  
- Labels restricted to the three options (5)  
- Applied to all relevant rows (2)

---

### Part 3.2 – Compare labels vs sentiment (10 marks)

Create a **cross-tabulation** (e.g., `pd.crosstab`) of dataset `label` vs `sentiment_label` and print it.

Add a short markdown commentary (3–4 sentences) describing:

- Where dataset labels and GPT sentiment appear to agree (e.g., label=1 mostly → Positive).  
- Any mismatches and at least one possible reason.

**Marks (10):**
- Correct cross-tab (6)  
- Sensible commentary on alignment and mismatches (4)


In [ ]:
# Part 3.1 – Define classify_sentiment and apply to summaries
# TODO: define classify_sentiment(summary: str) -> str using the chat API
# TODO: apply it to summarized_reviews and add a 'sentiment_label' column

def classify_sentiment(summary: str) -> str:

    prompt = (
        "Read this summarized review and classify its overall sentiment as one of: "
        "Positive, Neutral, or Negative. Return only the label."
        f"\n\nSummary: \"{summary}\""
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that classifies sentiment."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            stop=None,
        )
        label = response.choices[0].message['content'].strip()
        # Ensure label is one of the three allowed
        if label not in {"Positive", "Neutral", "Negative"}:
            label = "Neutral"
        return label
    except Exception as e:
        print(f"Error classifying sentiment: {e}")
        # On error, fallback to Neutral
        return "Neutral"


balanced_reviews=balanced_reviews.sort_values(by='summary', ascending=True)
variable_magic_number_view=10
balanced_reviews['sentiment_label'] = balanced_reviews[:variable_magic_number_view]['summary'].apply(classify_sentiment)

# Display the DataFrame with summaries
display(balanced_reviews.head(variable_magic_number_view))


Error classifying sentiment: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}
Error classifying sentiment: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}
Error classifying sentiment: Error code: 400 - {'error': {'message': "We could not parse the JSON body of yo

,label,title,content,summary,sentiment_label
17,1,Batman Begins,When I heard there was another Batman movie co...,Batman Begins defies expectations with strong ...,Neutral
53,1,What a fan of Clay Aiken would Love,If you liked Clay Aiken's style of singing on ...,"Clay Aiken's debut album, ""Measure of a Man,"" ...",Neutral
58,0,Just God Awful,Koontz turns the Frankenstein mythos into a co...,Koontz reimagines the Frankenstein myth as a c...,Neutral
36,1,In The Heart of the Family,Philbrick's award winning non-fiction retellin...,Philbrick's award-winning non-fiction retellin...,Neutral
13,1,An Old World change that's definitely wonderful,The Scarletti Curse is absolutely wonderful. M...,The Scarletti Curse by Ms. Feehan is a well-wr...,Neutral
57,0,Not nearly enough detail,It seems as if the author tried to get the boo...,The book appears rushed and lacks depth. It ma...,Neutral
96,1,Enjoyable Read,This book is engaging and really brings the st...,The book is engaging and vividly brings the st...,Neutral
14,1,"Manson Shows True Power Over His ""Family""",Manson shows it power and raps about his feeli...,The book offers a chilling and addictive accou...,Neutral
90,0,Overpriced for basic information,"This book is really full of common sense, but ...",The book offers common sense advice and compil...,Neutral
84,0,Bias Shows Through,While this book is very good on providing scri...,The book offers extensive scripture quotations...,Neutral


In [ ]:
# Part 3.2 – label vs sentiment cross-tab
# TODO: create a pd.crosstab of dataset label vs sentiment_label and print the result

crosstab_result = pd.crosstab(balanced_reviews['label'], balanced_reviews['sentiment_label'], rownames=['Original Label'], colnames=['Sentiment Label'])

# Print the crosstab
print(crosstab_result)



Sentiment Label  Neutral
Original Label          
0                      4
1                      6


**Part 3.2 – Short commentary (3–4 sentences):**  
*(Discuss agreement/disagreement between dataset labels and GPT sentiment, and why mismatches might happen.)*

---

## Part 4 – Mini “Review Assistant” Chatbot (15 marks)

You will build a very small **product Q&A assistant** that uses a handful of **summaries** as context.

### Part 4.1 – Prepare context (5 marks)

- Choose a small group of summaries (for example, 5–10 rows from `summarized_reviews`).  
- Combine them into a single **context string** that includes the label and summary for each review.

**Marks (5):**
- Reasonable subset and context construction (5)

---

### Part 4.2 – Implement `ask_review_assistant` (10 marks)

Write a function `ask_review_assistant(question: str) -> str` that:

- Uses the chat API with:
  - A **system message**, for example:  
    *"You are an assistant that answers questions about a product using ONLY the review summaries in the provided context. If the context does not contain enough information, say you don't know."*  
  - A **user message** that includes:
    - The context string (from Part 4.1)  
    - The user’s question

- Returns the model's answer.

Demonstrate the assistant by asking at least **3 different questions** about the product (e.g., durability, common complaints, suitability as a gift).

**Marks (10):**
- Correct system+user prompt design using the summaries as context (6)  
- At least 3 meaningful questions and answers (4)


In [ ]:
# Part 4.1 – Build a context from several summaries
# TODO: select 5–10 summarized reviews and build a single context string that includes label + summary.


variable_magic_number_view=10
string_combinedqna= balanced_reviews[:variable_magic_number_view]['summary']

display(string_combinedqna)
context_string = ""
for idx, row in string_combinedqna.items():
    context_string += f"Label {balanced_reviews.loc[idx, 'label']} Summary: {row}\n"

display(context_string)

context_string = str(context_string.strip())
display(context_string)


,summary
17,Batman Begins defies expectations with strong ...
53,"Clay Aiken's debut album, ""Measure of a Man,"" ..."
58,Koontz reimagines the Frankenstein myth as a c...
36,Philbrick's award-winning non-fiction retellin...
13,The Scarletti Curse by Ms. Feehan is a well-wr...
57,The book appears rushed and lacks depth. It ma...
96,The book is engaging and vividly brings the st...
14,The book offers a chilling and addictive accou...
90,The book offers common sense advice and compil...
84,The book offers extensive scripture quotations...


'Label 1 Summary: Batman Begins defies expectations with strong action and compelling plot development, offering a realistic and mature take compared to other superhero films. Featuring plot twists and a darker tone, it is suited for teens and adults. Overall, it\'s highly recommended as one of the best hero movies.\nLabel 1 Summary: Clay Aiken\'s debut album, "Measure of a Man," showcases his distinctive singing style with songs that suit his character and highlight his powerful voice. The tracks grow on listeners over time, making it a must-have for fans and a worthwhile purchase.\nLabel 0 Summary: Koontz reimagines the Frankenstein myth as a cop story filled with familiar clichés, including stereotypical characters and predictable dynamics. The most intriguing character, the first monster, is underdeveloped, and the story concludes with a disappointing ending.\nLabel 1 Summary: Philbrick\'s award-winning non-fiction retelling of the Essex tragedy compellingly explores family hardshi

'Label 1 Summary: Batman Begins defies expectations with strong action and compelling plot development, offering a realistic and mature take compared to other superhero films. Featuring plot twists and a darker tone, it is suited for teens and adults. Overall, it\'s highly recommended as one of the best hero movies.\nLabel 1 Summary: Clay Aiken\'s debut album, "Measure of a Man," showcases his distinctive singing style with songs that suit his character and highlight his powerful voice. The tracks grow on listeners over time, making it a must-have for fans and a worthwhile purchase.\nLabel 0 Summary: Koontz reimagines the Frankenstein myth as a cop story filled with familiar clichés, including stereotypical characters and predictable dynamics. The most intriguing character, the first monster, is underdeveloped, and the story concludes with a disappointing ending.\nLabel 1 Summary: Philbrick\'s award-winning non-fiction retelling of the Essex tragedy compellingly explores family hardshi

In [ ]:
from IPython.core.interactiveshell import dis
# Part 4.2 – Implement ask_review_assistant(question: str) -> str
# TODO: implement the chat call using the context and system prompt described above.

def ask_review_assistant(question: str) -> str:
    display(question)
    system_message = {
        "role": "system",
        "content": (
            "You are an assistant that answers questions about a product using ONLY the review summaries "
            "in the provided context. If the context does not contain enough information, say you don't know."
        )
    }

    user_message = {
        "role": "user",
        "content": f"Question: {question}"
    }

    response = client.chat.completions.create(
        model=model,
        messages=[system_message, user_message],
        temperature=0.2,
        max_tokens=250
    )

    display(response)
    answer = response.choices[0].message.content.strip()
    return answer

response=ask_review_assistant(context_string)
display(response)

'Label 1 Summary: Batman Begins defies expectations with strong action and compelling plot development, offering a realistic and mature take compared to other superhero films. Featuring plot twists and a darker tone, it is suited for teens and adults. Overall, it\'s highly recommended as one of the best hero movies.\nLabel 1 Summary: Clay Aiken\'s debut album, "Measure of a Man," showcases his distinctive singing style with songs that suit his character and highlight his powerful voice. The tracks grow on listeners over time, making it a must-have for fans and a worthwhile purchase.\nLabel 0 Summary: Koontz reimagines the Frankenstein myth as a cop story filled with familiar clichés, including stereotypical characters and predictable dynamics. The most intriguing character, the first monster, is underdeveloped, and the story concludes with a disappointing ending.\nLabel 1 Summary: Philbrick\'s award-winning non-fiction retelling of the Essex tragedy compellingly explores family hardshi

BadRequestError: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
# Part 4.2 – Demonstrate the assistant with at least 3 questions
# TODO: Call ask_review_assistant(...) three times with different questions and print the responses.

